# ToolRegistry + UXarray MCP — Power Demo

Shows `toolregistry` driving real UXarray mesh analysis tools using **gpt-5.5** from the local Argo gateway.

1. Register UXarray tools into a ToolRegistry
2. LLM picks and calls tools from natural-language requests
3. BM25 `discover_tools` — find the right tool from a large pool
4. Policy tag filtering — read-only safe session
5. Multi-turn agentic loop — LLM chains tool calls autonomously
6. Registry merge — combine UXarray + SimBoard tools in one session

In [1]:
import json
import os
import sys
import tempfile
from pathlib import Path

import numpy as np
import xarray as xr

# Make sure the uxarray-mcp source is importable
sys.path.insert(0, str(Path.home() / "uxarray-mcp-server/src"))

from openai import OpenAI
from toolregistry import ToolRegistry, ToolTag

# Local Argo gateway — OpenAI-compatible endpoint
CLIENT = OpenAI(base_url="http://localhost:44445/v1", api_key="argo")
MODEL = "argo:gpt-5.5"

import toolregistry

print(f"toolregistry {toolregistry.__version__}")
print(f"model: {MODEL}")

toolregistry 0.11.1
model: argo:gpt-5.5


In [2]:
# ── Synthetic mesh fixture ───────────────────────────────────────────────────


def make_mesh(n_faces=100):
    """Write a tiny UGRID mesh + data to a temp dir. Returns (grid_path, data_path)."""
    td = tempfile.mkdtemp(prefix="treg_demo_")
    n = n_faces + 2
    grid = xr.Dataset(
        {
            "Mesh2": (
                [],
                0,
                {
                    "cf_role": "mesh_topology",
                    "topology_dimension": 2,
                    "node_coordinates": "Mesh2_node_x Mesh2_node_y",
                    "face_node_connectivity": "Mesh2_face_nodes",
                },
            ),
            "Mesh2_node_x": (["nMesh2_node"], np.linspace(-180, 180, n)),
            "Mesh2_node_y": (["nMesh2_node"], np.linspace(-90, 90, n)),
            "Mesh2_face_nodes": (
                ["nMesh2_face", "nMaxMesh2_face_nodes"],
                [[i, i + 1, i + 2] for i in range(n_faces)],
                {"cf_role": "face_node_connectivity", "start_index": 0},
            ),
        }
    )
    data = xr.Dataset(
        {
            "temperature": (
                ["nMesh2_face"],
                np.random.uniform(260, 310, n_faces),
                {"units": "K", "long_name": "Surface Temperature"},
            ),
            "pressure": (
                ["nMesh2_face"],
                np.random.uniform(95000, 105000, n_faces),
                {"units": "Pa", "long_name": "Surface Pressure"},
            ),
        }
    )
    gp = os.path.join(td, "grid.nc")
    dp = os.path.join(td, "data.nc")
    grid.to_netcdf(gp)
    data.to_netcdf(dp)
    return gp, dp


GRID, DATA = make_mesh(100)
print("grid:", GRID)
print("data:", DATA)

grid: /var/folders/_f/dc3kjt3n3ms9ntb5fcj5mplm0000gq/T/treg_demo_82rq7afo/grid.nc
data: /var/folders/_f/dc3kjt3n3ms9ntb5fcj5mplm0000gq/T/treg_demo_82rq7afo/data.nc


## 1. Build a ToolRegistry from UXarray MCP tools

In [3]:
from uxarray_mcp.registry import build_registry

# core  = 31 visible tools (flat gateway names + namespaced session/hpc)
# deferred-full = 64 tools, 32 hidden until discovered via BM25
registry = build_registry(profile="core")

print(f"Visible tools: {len(registry.list_tools())}")
print()
for t in sorted(registry.list_tools()):
    print(f"  {t}")

Visible tools: 27

  analyze_dataset
  diagnose_endpoint
  get_capabilities
  get_result
  get_status
  hpc-endpoint_status
  hpc-get_execution_mode
  hpc-set_execution_mode
  hpc-validate_hpc_setup
  io-list_datasets
  manage_session
  plot_dataset
  probe_path_access
  prompt-first_look
  prompt-hpc_diagnose
  prompt-vorticity_analysis
  resume_workflow
  run_analysis
  run_workflow
  session-create_session
  session-get_operation_status
  session-get_result_handle
  session-get_session_state
  session-get_workflow_status
  session-list_operations
  session-register_dataset
  session-reset_session_state


## 2. LLM picks and calls a tool — zero orchestration code

In [4]:
def single_tool_call(prompt, reg=registry):
    """One-shot: LLM picks a tool and we execute it."""
    schemas = reg.get_schemas()
    resp = CLIENT.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        tools=schemas,
        tool_choice="auto",
    )
    msg = resp.choices[0].message
    if not msg.tool_calls:
        print("No tool call:", msg.content)
        return {}
    tc = msg.tool_calls[0]
    name = tc.function.name
    kwargs = json.loads(tc.function.arguments)
    print(f"→ {name}({json.dumps(kwargs, indent=2)})")
    fn = reg.get_callable(name)
    return fn(**kwargs)


# Inspect mesh topology
result = single_tool_call(
    f"Tell me the topology of this mesh (how many faces, nodes, edges): {GRID}"
)
print("\nResult:")
for k, v in result.items():
    if k not in ("_provenance", "recommended_next_steps"):
        print(f"  {k}: {v}")

→ run_analysis({
  "operation": "inspect_mesh",
  "grid_path": "/var/folders/_f/dc3kjt3n3ms9ntb5fcj5mplm0000gq/T/treg_demo_82rq7afo/grid.nc"
})

Result:
  format: UGRID
  n_face: 100
  n_node: 102
  n_edge: 201
  n_max_face_nodes: 3
  file_size_mb: 0.01


In [5]:
# Calculate face areas
result = single_tool_call(f"Compute the face areas for this mesh: {GRID}")
print("\nResult:")
for k, v in result.items():
    if k not in ("_provenance", "recommended_next_steps"):
        print(f"  {k}: {v}")

→ run_analysis({
  "operation": "calculate_area",
  "grid_path": "/var/folders/_f/dc3kjt3n3ms9ntb5fcj5mplm0000gq/T/treg_demo_82rq7afo/grid.nc"
})

Result:
  total_area: 0.0063896851367123185
  mean_area: 6.389685136712316e-05
  min_area: 2.813346695079885e-06
  max_area: 8.51299856757866e-05
  area_units: m^2
  n_face: 100


In [6]:
# Inspect variables
result = single_tool_call(
    f"What variables are in this dataset? grid={GRID} data={DATA}"
)
print("\nResult:")
for k, v in result.items():
    if k not in ("_provenance", "recommended_next_steps") and not isinstance(v, list):
        print(f"  {k}: {v}")

→ run_analysis({
  "operation": "inspect_variable",
  "grid_path": "/var/folders/_f/dc3kjt3n3ms9ntb5fcj5mplm0000gq/T/treg_demo_82rq7afo/grid.nc",
  "data_path": "/var/folders/_f/dc3kjt3n3ms9ntb5fcj5mplm0000gq/T/treg_demo_82rq7afo/data.nc"
})

Result:
  grid_info: {'n_face': 100, 'n_node': 102, 'n_edge': 201}


## 3. discover_tools — BM25 search across the full 58-tool pool

In [7]:
full = build_registry(profile="deferred-full")
discover = full.get_callable("discover_tools")

queries = [
    "compute vorticity wind curl",
    "subset bounding box region north atlantic",
    "calculate face areas",
    "compare two simulation fields bias rmse",
    "ensemble mean average members",
    "plot mesh wireframe visualization",
]

for q in queries:
    results = discover(query=q, top_k=3)
    print(f"\n'{q}'")
    for r in results:
        tag = "[deferred]" if r.get("deferred") else "[visible]"
        print(f"  {tag} {r['name']:45s} score={r['score']:.1f}")


'compute vorticity wind curl'
  [deferred] compute-calculate_curl                        score=26.1
  [deferred] compute-calculate_divergence                  score=20.8
  [deferred] compute-calculate_azimuthal_mean              score=8.2

'subset bounding box region north atlantic'
  [deferred] shape-subset_bbox                             score=38.9
  [deferred] shape-subset_polygon                          score=17.7
  [deferred] plot-plot_mesh_geo                            score=9.2

'calculate face areas'
  [deferred] compute-calculate_area                        score=17.5
  [deferred] compute-calculate_gradient                    score=7.6
  [deferred] compute-calculate_zonal_mean                  score=7.6

'compare two simulation fields bias rmse'
  [deferred] compute-compare_fields                        score=25.6
  [deferred] compute-calculate_rmse                        score=22.9
  [deferred] compute-calculate_bias                        score=22.8

'ensemble mean avera

## 4. Policy tag filtering — read-only safe session

In [8]:
# Build a registry that blocks anything touching the network or filesystem writes
safe = build_registry(profile="deferred-full")
safe.disable_by_tags([ToolTag.NETWORK, ToolTag.FILE_SYSTEM])

print(f"Full registry:  {len(full.list_tools())} tools")
print(f"Safe registry:  {len(safe.list_tools())} tools")
print()
print("Enabled in safe mode:")
for t in sorted(safe.list_tools()):
    print(f"  {t}")

Full registry:  58 tools
Safe registry:  35 tools

Enabled in safe mode:
  agent-run_scientific_agent
  compute-calculate_anomaly
  compute-calculate_bias
  compute-calculate_ensemble_mean
  compute-calculate_ensemble_spread
  compute-calculate_pattern_correlation
  compute-calculate_rmse
  compute-calculate_temporal_mean
  compute-compare_fields
  discover_tools
  get_capabilities
  get_result
  get_status
  hpc-get_execution_mode
  inspect-validate_dataset
  manage_session
  plot-plot_mesh_geo
  prompt-first_look
  prompt-hpc_diagnose
  prompt-vorticity_analysis
  resume_workflow
  run_workflow
  session-create_session
  session-get_operation_status
  session-get_result_handle
  session-get_session_state
  session-get_workflow_status
  session-list_operations
  session-register_dataset
  session-reset_session_state
  shape-extract_cross_section
  shape-regrid_dataset
  shape-remap_variable
  shape-subset_bbox
  shape-subset_polygon


## 5. Multi-turn agentic loop — LLM chains tool calls autonomously

In [9]:
def agentic_loop(task, reg=registry, max_turns=6):
    schemas = reg.get_schemas()
    messages = [
        {
            "role": "system",
            "content": "You are a climate mesh analysis assistant. Use tools to complete the task, then summarise.",
        },
        {"role": "user", "content": task},
    ]
    for turn in range(max_turns):
        resp = CLIENT.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=schemas,
            tool_choice="auto",
        )
        msg = resp.choices[0].message
        if not msg.tool_calls:
            print(f"[Turn {turn + 1}] Done.\n")
            return msg.content

        messages.append(msg)
        for tc in msg.tool_calls:
            name = tc.function.name
            kwargs = json.loads(tc.function.arguments)
            print(f"[Turn {turn + 1}] → {name}({list(kwargs.keys())})")
            fn = reg.get_callable(name)
            try:
                res = fn(**kwargs)
                if isinstance(res, dict):
                    res.pop("recommended_next_steps", None)
                content = json.dumps(res, default=str)[:3000]
            except Exception as e:
                content = f"Error: {e}"
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": content})
    return "Max turns reached."


answer = agentic_loop(
    f"""Analyse the mesh at {GRID} with data at {DATA}.
    I need: (1) mesh topology summary, (2) what variables exist and their stats,
    (3) face area statistics. Give a concise paragraph summary."""
)
print(answer)

[Turn 1] → analyze_dataset(['grid_path', 'data_path', 'include_plots'])


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


[Turn 2] → run_analysis(['operation', 'grid_path', 'data_path'])
[Turn 3] → run_analysis(['operation', 'grid_path'])
[Turn 4] Done.

The mesh is a small UGRID triangular mesh with 100 faces, 102 nodes, 201 edges, and a maximum of 3 nodes per face. The data file contains two face-centered variables, both valid with no NaNs/Infs/fill values: `temperature` (`n_face=100`, units K) ranges from 260.16 to 309.16 K with mean 284.06 K, and `pressure` (`n_face=100`, units Pa) ranges from 95,219.02 to 104,844.48 Pa with mean 100,558.93 Pa. Face area statistics over 100 faces are: total area 0.006389685 m², mean area 6.389685e-05 m², minimum 2.813347e-06 m², and maximum 8.512999e-05 m².


## 6. Registry merge — UXarray + SimBoard in one session

In [10]:
# Custom SimBoard stub tool
simboard = ToolRegistry(name="simboard")


def lookup_e3sm_run(case_name: str, machine: str = "perlmutter") -> dict:
    """Look up E3SM simulation metadata from SimBoard. Returns grid_name, compset, output_path.

    Args:
        case_name: E3SM case name (partial match ok).
        machine: HPC machine (default: perlmutter).
    """
    return {
        "case_name": case_name,
        "machine": machine,
        "grid_name": "ne30pg2_r05_IcoswISC30E3r5",
        "compset": "F2010xx-ZM",
        "status": "completed",
        "output_path": f"/pscratch/sd/w/whannah/{case_name}/run",
        "source": "SimBoard API (stub)",
    }


simboard.register(lookup_e3sm_run)
simboard.get_tool("lookup_e3sm_run").metadata.tags = {
    ToolTag.READ_ONLY,
    ToolTag.NETWORK,
}

# merge() mutates in-place; the registry name becomes a namespace prefix
combined = build_registry(profile="core")
combined.merge(simboard)  # tool lands as 'simboard-lookup_e3sm_run'

print(f"UXarray core:  {len(build_registry(profile='core').list_tools())} tools")
print(f"SimBoard:      {len(simboard.list_tools())} tools")
print(f"Combined:      {len(combined.list_tools())} tools")
print()
# Tool is namespaced: simboard-lookup_e3sm_run
simboard_tools = [t for t in combined.list_tools() if "simboard" in t]
print("SimBoard tools in combined:", simboard_tools)

UXarray core:  27 tools
SimBoard:      1 tools
Combined:      28 tools

SimBoard tools in combined: ['simboard-lookup_e3sm_run']


In [11]:
# One conversation spanning SimBoard lookup AND UXarray analysis
answer = agentic_loop(
    f"""Use simboard-lookup_e3sm_run to look up the E3SM case 'ZM-DEV-01' on perlmutter.
    Then inspect the mesh at {GRID} to get its topology.
    Report the grid_name from SimBoard and the face count from the mesh inspection.""",
    reg=combined,
)
print(answer)

[Turn 1] → simboard-lookup_e3sm_run(['case_name', 'machine'])
[Turn 1] → uxarray-run_analysis(['operation', 'grid_path'])
[Turn 2] Done.

SimBoard lookup and mesh inspection completed.

- **SimBoard grid_name:** `ne30pg2_r05_IcoswISC30E3r5`
- **Mesh face count:** `100`


## 7. Multi-provider schemas — same tools, different LLM APIs

In [12]:
# ToolRegistry generates schemas for OpenAI, Anthropic, Gemini from the same functions
mini = ToolRegistry(name="mini")


def get_mesh_summary(grid_path: str) -> dict:
    """Get topology summary of an unstructured mesh file."""
    from uxarray_mcp.tools import run_analysis

    return run_analysis(operation="inspect_mesh", grid_path=grid_path)


mini.register(get_mesh_summary)
mini.get_tool("get_mesh_summary").metadata.tags = {ToolTag.READ_ONLY}

openai_schema = mini.get_schemas()  # OpenAI tool_call format
print("OpenAI schema:")
print(json.dumps(openai_schema[0], indent=2))

# Same tool, different providers — zero code change:
# anthropic_schema = mini.get_schemas(format='anthropic')
# gemini_schema    = mini.get_schemas(format='gemini')

OpenAI schema:
{
  "type": "function",
  "function": {
    "name": "get_mesh_summary",
    "description": "Get topology summary of an unstructured mesh file.",
    "parameters": {
      "properties": {
        "grid_path": {
          "title": "Grid Path",
          "type": "string"
        }
      },
      "required": [
        "grid_path"
      ],
      "title": "get_mesh_summaryParameters",
      "type": "object"
    }
  }
}


## Summary

| Feature | FastMCP only | + ToolRegistry |
|---|---|---|
| Tool schemas | MCP wire only | OpenAI / Anthropic / Gemini / MCP |
| Tool discovery | LLM sees all tools | BM25 `discover_tools` from deferred pool |
| Policy control | None | `ToolTag` + `disable_by_tags()` |
| Registry merge | None | `registry.merge(other)` |
| Admin panel | None | Built-in web UI (port 8081) |
| Multi-transport | stdio only | stdio / SSE / HTTP / OpenAPI |

The UXarray domain tools don't change. Only the control plane around them.